# Suite P3PRS — Parsing Python 3 into the store

`Python3.Parse.parse(text, path)` reads Python 3 source with tree-sitter-python and gives the callback of the
transaction that writes it: the module, filed in the workspace by its path, and its statements, owned through
`Programs.Framework.HasParent` by property and index, each with its line. Expressions are mbse-expressions' `Python3`
terms, annotations and type parameters `Python Types` terms, case patterns `Python Patterns` terms, each keeping the
concrete syntax the dialect holds. Each case parses constructs into one store, reads the statements back as plain data
within a transaction (a term as `ToText` writes it, or as its kind, attributes and subterms), and checks them; then
invalid syntax, which is kept, and the one error, text that is no text, with its line and column.

In [ ]:
import ast
import itertools
import json
from pathlib import Path

from mbse.Expressions import Framework as Expressions
from mbse.Expressions.Framework import Objects, Terms
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Expressions as P
from mbse.Expressions.Python.Python3.Text import ToText
from mbse.Programs import Python3 as Programs
from mbse.Programs.Framework import Model
from mbse.Programs.Framework.Errors import ParseError
from mbse.Programs.Python3 import Parse
from mbse.Programs.Python3.Statements import KINDS
from mbse.Schemas.Framework import Builders, Plain
from support import raises

SOURCES = Path("../../conformance/sources")
INSTALLS = (Expressions.install, ISOIEC11404.install, IEEE754.install, Python3.install, PythonTypes.install,
            PythonPatterns.install, Model.install, Programs.install)


def new_store():
    store = Builders.Store()
    for install in INSTALLS:
        store.transaction(install)
    return store


STORE = new_store()
PATHS = (f"case{n}.py" for n in itertools.count())


def shape(term):
    """A term as plain data: its kind, its attributes (a helper as its kind and attributes), and its subterms."""
    def plain(value):
        if isinstance(value, Terms.Part):
            return (value.kind, {k: plain(v) for k, v in value.attributes.items()})
        return [plain(v) for v in value] if isinstance(value, tuple) else value
    tag, record = Terms.kind_of(term)
    return (tag, {k: plain(v) for k, v in Terms.attributes_of(record).items()},
            [None if a is None else shape(a) for a in Terms.arguments_of(record)])


def plain(value, shapes):
    """What a property holds, as plain data: a term as its text (or its `shape`), a container as a list, a value object
    as a dict of its properties."""
    if isinstance(value, Objects.Listed):
        return [plain(v, shapes) for v in value.values]
    if Terms.is_term(value):
        return shape(value) if shapes else ToText(value)
    if hasattr(value, "as_schema"):
        return {k: plain(v, shapes) for k, v in Objects.properties(value).items()}
    return value


def statement(node, shapes):
    """A statement as plain data: its kind, its line, its properties, and its children by property, in order."""
    found = {"kind": node.as_schema().name.rsplit(".", 1)[-1], "line": next(iter(node.location)).line.get(),
             **{k: plain(v, shapes) for k, v in Objects.properties(node).items()}}
    for entry in sorted(node.children, key=lambda e: (e.property.get(), e.index.get())):
        found.setdefault(entry.property.get(), []).append(statement(entry.child, shapes))
    return found


def parse(text, shapes=False):
    """The statements of the module `text` parses to, written in the store at a path of its own, read back."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    with STORE.transaction():
        module = STORE.singleton(Model.WORKSPACE).files({"path": path})
        return [statement(e.child, shapes) for e in sorted(module.body, key=lambda e: e.index.get())]


def first(text, shapes=False):
    return parse(text, shapes)[0]


def value(text, shapes=False):
    """The expression of the expression statement `text`."""
    return first(text, shapes)["value"]


def kinds(statements):
    return [s["kind"] for s in statements]


def fails(text, message, line, column):
    with raises(ParseError, match=message):
        Parse.parse(text, "failing.py")
    try:
        Parse.parse(text, "failing.py")
    except ParseError as error:
        assert (error.line, error.column) == (line, column), (error.line, error.column)

## P3PRS-01 · The conformance source is one transaction: every statement kind, terms the dialects validate, and JSON that reads back and writes the same bytes

In [ ]:
text = (SOURCES / "python314.py").read_text(encoding="utf-8")
(frame,) = STORE.transaction(Parse.parse(text, "python314.py"))
assert {"ws", "module"} <= set(frame) and len(frame) == 2 + 151  # the workspace, the module, each statement
seen, problems = set(), []


def terms(held):
    if isinstance(held, Objects.Listed):
        for v in held.values:
            yield from terms(v)
    elif Terms.is_term(held):
        yield held
    elif hasattr(held, "as_schema"):
        for v in Objects.properties(held).values():
            yield from terms(v)


def visit(node):
    seen.add(node.as_schema().name.rsplit(".", 1)[-1])
    for term in terms(node):  # names are bound by the program, not the term: only other problems count
        problems.extend(p for p in P.DIALECT.validate(term) if "is not bound" not in p)
    for entry in node.children:
        visit(entry.child)


with STORE.transaction():
    module = STORE.singleton(Model.WORKSPACE).files({"path": "python314.py"})
    for entry in module.body:
        visit(entry.child)
assert seen == set(KINDS), set(KINDS) - seen
assert problems == [], problems[:5]
# Written as plain JSON, read into a fresh store holding the installs, executed, and written again: the same bytes
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
    Parse.parse(text, "python314.py", STORE))))
fresh = new_store()
(again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
assert sorted(again) == sorted(frame)
assert json.dumps(Plain.ToPlain(fresh).Transaction(fresh.Mutations.OfTransaction(
    Parse.parse(text, "python314.py", fresh)))) == written
with fresh.transaction():
    assert len(list(fresh.singleton(Model.WORKSPACE).files({"path": "python314.py"}).body)) == 25
print(len(seen), "kinds,", len(written), "bytes")

## P3PRS-02 · The transaction: one upsert by the file's path, the module filed by path, statements linked by property and index, each with its line

In [ ]:
(frame,) = STORE.transaction(Parse.parse("if a:\n    pass\nelse:\n    b = 1\n", "shape.py"))
assert sorted(frame) == ["module", "s0", "s1", "s2", "ws"]  # in source order
with STORE.transaction():
    module = STORE.singleton(Model.WORKSPACE).files({"path": "shape.py"})
    assert module.as_schema().name == "Programs.Python3.Module"
    (top,) = module.body
    assert (top.property.get(), top.index.get(), top.child.as_schema().name) == ("body", 0, "Programs.Python3.If")
    links = sorted((e.property.get(), e.index.get(), e.child.as_schema().name) for e in top.child.children)
    assert links == [("body", 0, "Programs.Python3.Pass"), ("orelse", 0, "Programs.Python3.Assign")]
    assert [(e.line.get(), e.file.as_schema().name) for e in top.child.location] == [(1, "Programs.Python3.Module")]
# Run again, the upsert finds what it wrote
STORE.transaction(Parse.parse("if a:\n    pass\nelse:\n    b = 1\n", "shape.py"))
with STORE.transaction():
    assert len(list(STORE.singleton(Model.WORKSPACE).files({"path": "shape.py"}).body)) == 1
# The callback names its schemas through what store.transaction gives it; a bare builder needs the store
with raises(TypeError):
    STORE.Mutations.OfTransaction(Parse.parse("pass", "bare.py"))
built = Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse("pass", "bare.py", STORE)))
assert "bare.py" in json.dumps(built)
assert parse("") == [] and kinds(parse("x\n\n\ny\n")) == ["Expr", "Expr"]
assert [s["line"] for s in parse("x = 1\n\n@d\ndef f():\n    pass\n")] == [1, 4]  # a definition's line, as `ast`'s
print("ok")

## P3PRS-03 · Literals keep their spelling: numbers, strings and bytes, prefixes and quotes, and adjacent strings

In [ ]:
def literal(text):
    tag, attributes, _ = value(text, shapes=True)
    return tag, attributes


assert literal("0x_FF") == ("literal", {"value": 255, "text": "0x_FF", "base": 16, "separators": True})
assert literal("0o17")[1]["base"] == 8 and literal("0b1010")[1]["base"] == 2 and literal("10")[1]["base"] == 10
assert literal("1.5e-3") == ("literal", {"value": 0.0015, "text": "1.5e-3"})
assert literal("True") == ("literal", {"value": True, "text": "True"})
assert literal("None") == ("constant", {"singleton": "None", "text": "None"})
assert literal("...") == ("constant", {"singleton": "...", "text": "..."})
tag, attributes = literal("3.0J")
assert tag == "constant" and attributes["text"] == "3.0J" and attributes["complex"][1] == {"real": 0.0, "imag": 3.0}
assert literal("rb'\\d'") == ("literal", {"value": b"\\d", "text": "rb'\\d'", "prefix": "rb", "quote": "'"})
assert literal('"""a\nb"""') == ("literal", {"value": "a\nb", "text": '"""a\nb"""', "quote": '"""'})
assert literal("'\\n\\u00e9'")[1]["value"] == "\n\u00e9"
for text in ("0", "1_000", "0x_FF", "0o17", "0b1010", "1.5e-3", "2j", "3.0J", ".5", "5.", "None", "True", "False",
             "...", "'a'", '"a"', "'''a'''", '"""a\nb"""', "b'a'", "rb'\\d'", "R'a'", "u'a'", "'\\n\\u00e9'"):
    assert value(text) == text, (text, value(text))
concatenated = value("'a' \"b\" f'{c}'", shapes=True)
assert concatenated[0] == "concatenatedstring" and [t[0] for t in concatenated[2]] == ["literal", "literal", "joinedstr"]
assert value("('a'\n 'b')") == "('a' 'b')"  # parentheses kept, layout the printer's
print("ok")

## P3PRS-04 · f-strings and t-strings: their text, replacement fields, conversions, format specs and self-documenting fields

In [ ]:
tag, joined, fields = value("f'a{b!r:>{w}} {c = } {{d}}'", shapes=True)
assert tag == "joinedstr" and (joined["prefix"], joined["quote"]) == ("f", "'")
assert [part[0] for part in joined["values"]] == ["stringtext", "formattedvalue", "stringtext", "formattedvalue",
                                                  "stringtext"]
field = joined["values"][1][1]
assert (field["text"], field["conversion"]) == ("b", "r") and "debug" not in field
assert field["format_spec"][1]["values"] == [("stringtext", {"spelling": ">"}), ("formattedvalue", {"text": "w"})]
assert joined["values"][3][1] == {"text": "c = ", "debug": True} and joined["values"][4][1]["spelling"] == " {{d}}"
assert [f[1]["name"] for f in fields] == ["b", "w", "c"]  # the fields' expressions, in order
tag, template, _ = value("t'{x!s:{y}}'", shapes=True)
assert tag == "templatestr" and template["values"][0][0] == "interpolation"
assert template["values"][0][1]["format_spec"][1]["values"][0][0] == "formattedvalue"
assert value("rf'\\\\'", shapes=True)[1]["values"] == [("stringtext", {"spelling": "\\\\"})]  # not the quote's
spec = value('f"{x:=10}"', shapes=True)[1]["values"][0][1]  # a spec, not an assignment
assert spec["format_spec"][1]["values"] == [("stringtext", {"spelling": "=10"})] and spec["text"] == "x"
assert value('f"{(x:=10)}"', shapes=True)[2][0][0] == "parenthesized"
assert value('F"""\n{x}\n"""', shapes=True)[1]["quote"] == '"""' and "values" not in value("Rf''", shapes=True)[1]
for text in ("f'a{b!r:>{w}} {c = } {{d}}'", "t'{x!s:{y}}'", "rf'\\\\'", 'f"{x:=10}"', 'f"{x = !r:^10}"',
             "f'{\"nested\"}'", 'f"{f\'{x}\'}"', "f'{x,y}'", "f'{a, *b}'", "f'{yield x}'"):
    assert value(text) == text, (text, value(text))
print("ok")

## P3PRS-05 · Expressions, with Python's precedence where tree-sitter-python's differs, and parentheses as written

In [ ]:
def operators(term):
    tag, attributes, subterms = term
    return (attributes.get("op"), *[operators(s) for s in subterms if s is not None and s[0] == "binop"])


regrouped = value("~a ^ b & c | d & e ^ -f", shapes=True)
assert operators(regrouped) == ("|", ("^", ("&",)), ("^", ("&",)))  # `&` binds tighter than `^`
assert value("~a ^ b & c | d & e ^ -f") == "~a ^ b & c | d & e ^ -f"
assert operators(value("a ** b ** c", shapes=True)) == ("**", ("**",))  # to the right
assert value("a - b - c", shapes=True)[2][0][0] == "binop"  # to the left
awaited = value("await x ** 2", shapes=True)
assert awaited[0] == "binop" and awaited[2][0][0] == "await"  # `(await x) ** 2`
walrus = value("(m := a if b else c)", shapes=True)
assert walrus[0] == "parenthesized" and walrus[2][0][0] == "namedexpr" and walrus[2][0][2][0][0] == "ifexp"
chain = value("a < b <= c is not d not in e", shapes=True)
assert [link[1]["op"] for link in chain[1]["comparisons"]] == ["<", "<=", "is not", "not in"]
assert value("a is  not b") == "a is not b"
assert len(value("a and b and c", shapes=True)[2]) == 3 and len(value("a and (b and c)", shapes=True)[2]) == 2
assert value("not a", shapes=True)[1] == {"op": "not"}
call = value("f(a, *b, c=1, **d)", shapes=True)
assert [k[1].get("arg") for k in call[1]["keywords"]] == ["c", None] and call[2][2][0] == "starred"
assert value("f(x for x in y)", shapes=True)[2][1][0] == "generatorexp"
index = value("a[1:2, ::3, *s]", shapes=True)[2][1]
assert index[0] == "tuple" and [t[0] for t in index[2]] == ["slice", "slice", "starred"]
assert value("a[1,]", shapes=True)[2][1][0] == "tuple" and value("a[1]", shapes=True)[2][1][0] == "literal"
assert value("()", shapes=True)[0] == "tuple" and value("(a, b)", shapes=True)[0] == "tuple"  # its own parentheses
assert value("a, b", shapes=True)[0] == "tuple" and value("a,") == "(a,)"  # bare: a tuple too, which ToText parenthesizes
assert value("(a)", shapes=True)[0] == "parenthesized" and value("((a, b))", shapes=True)[0] == "parenthesized"
assert value("{**d for d in ds}", shapes=True)[0] == "dictcomp"
lam = value("lambda x, /, y=1, *z, w, **v: x", shapes=True)[1]["args"][1]
assert [p[1]["arg"] for p in lam["posonlyargs"]] == ["x"] and lam["vararg"][1]["arg"] == "z"
for text in ("a if b else c", "lambda: 0", "[x async for x in y if x if not x]", "{k: v for k, v in d}",
             "{**a, 'b': 1}", "{1, *a}", "a.b(c)[d:e]", "-a ** -b", "(yield)", "x[...]", "f(*a, *b)"):
    assert value(text) == text, (text, value(text))
print("ok")

## P3PRS-06 · Assignment targets, and annotated and augmented assignment

In [ ]:
assign = first("a = b = c")
assert assign["kind"] == "Assign" and assign["targets"] == ["a", "b"] and assign["value"] == "c"
assert first("a, *b = c", shapes=True)["targets"][0][0] == "tuple"
assert first("[a, b] = c", shapes=True)["targets"][0][0] == "list"
assert first("(a) = c", shapes=True)["targets"][0][0] == "parenthesized"
assert first("() = c", shapes=True)["targets"][0] == ("tuple", {}, [])
assert [first(f"x {op}= 1")["op"] for op in ("+", "//", "**", "@", "<<")] == ["+", "//", "**", "@", "<<"]
annotated = first("x: list[int] | None = None")
assert annotated["kind"] == "AnnAssign" and annotated["annotation"] == "list[int] | None" and "value" not in first("x: int")
assert first("(x): int", shapes=True)["target"][0] == "parenthesized"
assert first("a.b: int", shapes=True)["target"][0] == "attribute"
renamed = first("type(t).__name__ = 'x'")  # an assignment, which tree-sitter-python reads as a type alias
assert renamed["kind"] == "Assign" and renamed["targets"] == ["type(t).__name__"]
union = first("def f(a: list[str] | str | None): pass", shapes=True)["args"]["args"][0]["annotation"]
assert union[1]["op"] == "|" and union[2][0][1]["op"] == "|"  # `|` groups to the left
assert first("a = yield b")["value"] == "(yield b)" and first("for (a, b), c in d: pass")["target"] == "((a, b), c)"
print("ok")

## P3PRS-07 · Simple statements: imports and their aliases, global and nonlocal, del, assert, raise, return and type aliases

In [ ]:
assert first("import a.b.c as d, e")["names"] == [{"name": "a.b.c", "asname": "d"}, {"name": "e"}]
relative = first("from ..m import x as y, z")
assert (relative["level"], relative["module"], len(relative["names"])) == (2, "m", 2)
assert "module" not in first("from . import x") and first("from ... import x")["level"] == 3
assert first("from m import *")["names"] == [{"name": "*"}] and "level" not in first("from m import x")
assert first("from __future__ import annotations as an") == {
    "kind": "ImportFrom", "line": 1, "module": "__future__", "names": [{"name": "annotations", "asname": "an"}]}
assert first("global a, b")["names"] == ["a", "b"]
assert first("def f():\n    nonlocal a")["body"][0] == {"kind": "Nonlocal", "line": 2, "names": ["a"]}
assert first("del a, b[0], c.d")["targets"] == ["a", "b[0]", "c.d"]
assert first("assert a, 'm'") == {"kind": "Assert", "line": 1, "test": "a", "msg": "'m'"}
assert first("raise") == {"kind": "Raise", "line": 1} and first("raise E from c")["cause"] == "c"
assert first("return") == {"kind": "Return", "line": 1} and first("return a, b")["value"] == "(a, b)"
alias = first("type A[T: int, *Ts, **P] = Callable[P, tuple[T, *Ts]]", shapes=True)
assert alias["kind"] == "TypeAlias" and alias["name"] == ("variable", {"name": "A"}, [])
assert [t[0] for t in alias["type_params"]] == ["typevar", "typevartuple", "paramspec"]
assert first("type A = B | C | D")["value"] == "B | C | D"
assert kinds(parse("pass\nbreak\ncontinue\nx\n")) == ["Pass", "Break", "Continue", "Expr"]
assert first("print >> f, x")["value"] == "(print >> f, x)"  # Python 3 too: a tuple of a shift and a value
print("ok")

## P3PRS-08 · Compound statements: conditions, loops, exceptions, context managers and matches, their clauses as children by property

In [ ]:
chain = first("if a:\n    pass\nelif b:\n    pass\nelif c:\n    pass\nelse:\n    pass")
assert chain["orelse"][0]["kind"] == "If" and chain["orelse"][0]["test"] == "b"
assert kinds(chain["orelse"][0]["orelse"][0]["orelse"]) == ["Pass"] and chain["orelse"][0]["line"] == 3
loop = first("while a:\n    pass\nelse:\n    pass")
assert (kinds(loop["body"]), kinds(loop["orelse"])) == (["Pass"], ["Pass"])
assert first("async def f():\n    async for a in b:\n        pass")["body"][0]["kind"] == "AsyncFor"
tried = first("try:\n    pass\nexcept (A, B) as e:\n    pass\nexcept C:\n    pass\nexcept:\n    pass\n"
              "else:\n    pass\nfinally:\n    pass")
assert tried["kind"] == "Try" and kinds(tried["handlers"]) == ["ExceptHandler"] * 3
assert [h.get("name") for h in tried["handlers"]] == ["e", None, None]
assert [h.get("type") for h in tried["handlers"]] == ["(A, B)", "C", None]
assert kinds(tried["orelse"]) == kinds(tried["finalbody"]) == ["Pass"]
assert first("try:\n    pass\nexcept A, B:\n    pass")["handlers"][0]["type"] == "(A, B)"  # bare: a tuple too
assert first("try:\n    pass\nexcept* A:\n    pass")["kind"] == "TryStar"
with_ = first("with a as b, c:\n    pass")
assert with_["items"] == [{"context_expr": "a", "optional_vars": "b"}, {"context_expr": "c"}]
assert first("with (a as b):\n    pass")["items"] == [{"context_expr": "a", "optional_vars": "b"}]
assert first("async def f():\n    async with a:\n        pass")["body"][0]["kind"] == "AsyncWith"
match = first("match a, b:\n    case 1:\n        pass\n    case _ if c:\n        pass")
assert match["subject"] == "(a, b)" and kinds(match["cases"]) == ["MatchCase", "MatchCase"]
assert match["cases"][1]["guard"] == "c" and kinds(match["cases"][0]["body"]) == ["Pass"]
assert kinds(first("if a: b = 1; c = 2")["body"]) == ["Assign", "Assign"]
print("ok")

## P3PRS-09 · Definitions: functions, classes, their decorators, parameters (`Arguments`), keywords and type parameters (`Python Types`)

In [ ]:
function = first("@d\n@e.f(1)\nasync def f[T](a, /, b: int = 1, *c: *Ts, d, e=2, **f: int) -> T:\n    pass")
assert function["kind"] == "AsyncFunctionDef" and function["decorator_list"] == ["d", "e.f(1)"]
assert function["args"] == {"posonlyargs": [{"arg": "a"}], "args": [{"arg": "b", "annotation": "int", "default_value": "1"}],
                            "vararg": {"arg": "c", "annotation": "*Ts"}, "kwonlyargs": [{"arg": "d"}, {"arg": "e", "default_value": "2"}],
                            "kwarg": {"arg": "f", "annotation": "int"}}
assert function["returns"] == "T" and function["type_params"] == ["T"]
assert first("def f(): pass")["args"] == {} and first("def f(*, a): pass")["args"] == {"kwonlyargs": [{"arg": "a"}]}
generic = first("class C[T: int, U: (int, str), *Ts, **P](B, metaclass=M, **k):\n    pass", shapes=True)
assert [(t[0], t[1]["name"]) for t in generic["type_params"]] == [
    ("typevar", "T"), ("typevar", "U"), ("typevartuple", "Ts"), ("paramspec", "P")]
assert generic["type_params"][1][2][0][0] == "tuple"  # its bound
assert [b[0] for b in generic["bases"]] == ["variable"] and [k.get("arg") for k in generic["keywords"]] == ["metaclass", None]
assert "bases" not in first("class C:\n    pass") and "bases" not in first("class C():\n    pass")
assert first("@(lambda f: f)\ndef f():\n    pass")["decorator_list"] == ["(lambda f: f)"]
assert kinds(first("class C(B):\n    x: int = 0\n    def m(self): pass")["body"]) == ["AnnAssign", "FunctionDef"]
print("ok")

## P3PRS-10 · Patterns (`Python Patterns`): values, singletons, sequences, mappings, classes, captures, alternatives and groups

In [ ]:
def case(pattern):
    return first(f"match x:\n    case {pattern}:\n        pass", shapes=True)["cases"][0]["pattern"]


assert case("1")[0] == "matchvalue" and case("-1")[2][0][1] == {"op": "-"} and case("-1 + 2j")[2][0][1] == {"op": "+"}
assert case("a.b")[0] == "matchvalue" and case("a") == ("matchas", {"name": "a"}, [None])
assert case("_") == ("matchas", {}, [None])
assert [case(s)[1]["value"] for s in ("None", "True", "False")] == ["None", "True", "False"]
assert [case(s)[1].get("delimiters") for s in ("[a]", "(a,)", "a, b", "()")] == ["[]", "()", None, "()"]
assert case("(a)")[0] == "matchas"  # a group is its pattern
assert [p[0] for p in case("1 | (2 | 3)")[2]] == ["matchvalue", "matchor"] and len(case("1 | 2 | 3")[2]) == 3
assert [p[0] for p in case("-1 | _")[2]] == ["matchvalue", "matchas"]
mapping = case("{'k': v, -1: _, **rest}")
assert mapping[1]["rest"] == "rest" and [t[0] for t in mapping[2]] == ["literal", "unaryop", "matchas", "matchas"]
assert [t[0] for t in case("{a.b: x, 1 + 2j: y}")[2][:2]] == ["attribute", "binop"]
cls = case("P(1, x=-2, y=_, z=a.b, w=str() as s)")
assert cls[0] == "matchclass" and cls[2][0] == ("variable", {"name": "P"}, [])
assert [p[0] for p in cls[2][1:]] == ["matchvalue", "matchvalue", "matchas", "matchvalue", "matchas"]
assert cls[2][-1][1]["name"] == "s" and cls[2][-1][2][0][0] == "matchclass"
assert case("[*a, *_]")[2][1] == ("matchstar", {}, []) and case("(1 | 2) as s")[1]["name"] == "s"
for pattern in ("1", "-1", "-1 + 2j", "1.5 - 2j", "'a' 'b'", "b'a'", "a.b.c", "a", "_", "None", "[a, *b]", "(a,)",
                "()", "[]", "{'k': v, -1: _, **rest}", "{}", "{a.b: x, -1 - 2j: y}", "P(1, x=-2, y=_, z=a.b, w=str() as s)",
                "P()", "1 | 2 | 3", "1 | (2 | 3)", "1 | 2 as s", "[a, b] as c", "[(a as b) as c]", "a, b", "a,", "[a | b]"):
    written = first(f"match x:\n    case {pattern}:\n        pass")["cases"][0]["pattern"]
    assert written == pattern, (pattern, written)
print("ok")

## P3PRS-11 · Comments are statements where statements are listed, on their own lines or trailing; elsewhere they are dropped

In [ ]:
module = parse("# lead\nx = 1  # trailing\nif a:  # header\n    pass\n    # end of block\n# between\nelif b:\n    pass\n"
               "else:\n    # first\n    pass\ny = (1,  # dropped\n     2)\n")


def comments(statements):
    for s in statements:
        if s["kind"] == "Comment":
            yield s["text"], s.get("trailing", False)
        for prop in ("body", "orelse", "handlers", "finalbody", "cases"):
            yield from comments(s.get(prop, []))


assert list(comments(module)) == [(" lead", False), (" trailing", True), (" header", True), (" end of block", False),
                                  (" between", False), (" first", False)]
assert kinds(module[3]["body"]) == ["Comment", "Pass", "Comment", "Comment"] and module[1]["line"] == 2
match = first("match a:\n    case 1:\n        pass\n    # between cases\n    case 2:\n        pass\n")
assert kinds(match["cases"][0]["body"]) == ["Pass", "Comment"]
tried = first("try:\n    pass\n# before except\nexcept E:\n    pass\n# before finally\nfinally:\n    pass\n")
assert kinds(tried["body"]) == ["Pass", "Comment"] and kinds(tried["handlers"][0]["body"]) == ["Pass", "Comment"]
assert kinds(parse("x = 1 \\\n    + 2")) == ["Assign"]  # line continuations are layout
print("ok")

## P3PRS-12 · Python 3.13 and later, which the pre-pass puts back: lazy imports, dict comprehensions that unpack, and type parameters' defaults

In [ ]:
lazy = parse("lazy import json\nlazy from . import x\nimport os\n")
assert [s.get("is_lazy", False) for s in lazy] == [True, True, False]
unpacked = value("{**d for d in ds}", shapes=True)
assert unpacked[0] == "dictcomp" and unpacked[2][1] is None and unpacked[2][2][0] == "variable"  # no value; its key
assert value("{**d for d in ds}") == "{**d for d in ds}"
assert value("{**a, **b}", shapes=True)[0] == "dict"  # a display, not a comprehension
assert parse("{**a for a in b}\n[{**c} for c in d]")[0]["value"] == "{**a for a in b}"
defaults = first("def f[T = int, *Ts = (int,), **P = [str], U: str = 'u'](): pass", shapes=True)["type_params"]
assert [d[2][-1][0] for d in defaults] == ["variable", "tuple", "list", "literal"]  # each default, last
assert defaults[3][2][0] == ("variable", {"name": "str"}, [])  # a bound, then a default
assert first("class C[T = dict[str,\n    int]]: pass")["type_params"] == ["T = dict[str, int]"]
assert first("type A[T = int] = list[T]")["type_params"] == ["T = int"]
assert parse("lazy = 1\nlazy(import_)\n")[0]["targets"] == ["lazy"]  # still a name
assert parse("lazy import x\ny = {**a, 'b': 1}\n")[1]["value"] == "{**a, 'b': 1}"
print("ok")

## P3PRS-13 · Invalid syntax is kept, never raised: an `Error` where a statement stands, an `error` or `missing` term within an expression, each as written; only text that is no text raises

In [ ]:
# Where a statement stands: an `Error`, its text as written, its line, and the statements recovered within it
mixed = parse("a = 1\nx = = 2\nb = 3\n")
assert kinds(mixed) == ["Assign", "Error", "Assign"] and mixed[1] == {"kind": "Error", "line": 2, "text": "x = = 2"}
assert first("x = (") == {"kind": "Error", "line": 1, "text": "x = ("}  # what tree-sitter skipped, `ERROR`
assert first("def f(:") == {"kind": "Error", "line": 1, "text": "def f(:"}  # a `MISSING` `)`, which no term holds
recovered = first("class C(:\n    pass\n    y = 1\n")
assert recovered["text"] == "class C(:\n    pass\n    y = 1" and kinds(recovered["body"]) == ["Pass", "Assign"]
assert recovered["body"][1]["line"] == 3
nested = first("def f():\n    x = = 1\n    y = 2\n")
assert kinds(nested["body"]) == ["Error", "Assign"] and nested["body"][0] == {"kind": "Error", "line": 2, "text": "x = = 1"}
tried = first("try:\n    pass\nexcept E +:\n    pass\n")  # a clause is no statement: its statement is invalid whole
assert tried["kind"] == "Error" and kinds(tried["body"]) == ["Pass", "Pass"]
cased = first("match x:\n    case 1 +:\n        pass")
assert cased["kind"] == "Error" and cased["body"][0]["text"] == "case 1 +:\n        pass"
assert kinds(cased["body"][0]["body"]) == ["Pass"]
assert parse("lazy import json\nx = (\nlazy import os\n")[1]["text"] == "x = ("  # around what the pre-pass rewrote
assert first("lazy import (")["text"] == "lazy import ("  # as written, not as the pre-pass rewrote it
# Python 2's statements, and constructs the dialects cannot hold
assert first("print 'x'") == {"kind": "Error", "line": 1, "text": "print 'x'"}
assert first("exec 'x'") == {"kind": "Error", "line": 1, "text": "exec 'x'"}
tuple_parameter = first("def f((a, b)): pass")
assert tuple_parameter["text"] == "def f((a, b)): pass" and kinds(tuple_parameter["body"]) == ["Pass"]
for text in ("def f((a, b)=1): pass", "def f[a.b](): pass", "def f[T = ](): pass", "def f[T = (](): pass"):
    assert first(text) == {"kind": "Error", "line": 1, "text": text, "body": [{"kind": "Pass", "line": 1}]}, text
decorated = first("@d\ndef f((a, b)): pass")  # a statement within one, recovered as one, itself invalid
assert decorated["text"] == "@d\ndef f((a, b)): pass" and decorated["body"] == [
    {"kind": "Error", "line": 2, "text": "def f((a, b)): pass", "body": [{"kind": "Pass", "line": 2}]}]
# A newline found missing, which tree-sitter's API does not show: an `Error` with no text
assert parse("return return") == [{"kind": "Return", "line": 1}, {"kind": "Error", "line": 1, "text": ""},
                                  {"kind": "Return", "line": 1}]
assert kinds(parse("a; b")) == ["Expr", "Expr"] and kinds(first("if a: b")["body"]) == ["Expr"]
# Within an expression: an `error` term, the innermost expression that holds it, with the terms recovered within it
assert value("(1 + )", shapes=True) == ("error", {"text": "(1 + )"}, [("literal", {"value": 1, "text": "1", "base": 10}, [])])
assert value("f(a,,b)", shapes=True)[1:] == ({"text": "f(a,,b)"}, [("variable", {"name": n}, []) for n in "fab"])
assert value("a <> b", shapes=True) == ("error", {"text": "a <> b"}, [("variable", {"name": "a"}, []),
                                                                    ("variable", {"name": "b"}, [])])
assert value("10L", shapes=True) == ("error", {"text": "10L"}, [])  # a literal CPython does not read
assert value("(+)", shapes=True) == ("error", {"text": "(+)"}, [])
assert value("(a as b)", shapes=True)[2][0][:2] == ("error", {"text": "a as b"})
assert value("lambda (a, b): 0", shapes=True)[1] == {"text": "lambda (a, b): 0"}
assert [t[0] for t in value("a[*b, *(c)]", shapes=True)[2]] == ["variable", "starred", "parenthesized"]  # 0.25 lacks it
assert first("x = (1,\n     é é)")["value"] == "(1,\n     é é)"  # as written, printed by `ToText`
assert first("\U0001d465 = (1,\n     \U0001d465 \U0001d465)", shapes=True)["value"][1] == {
    "text": "(1,\n     \U0001d465 \U0001d465)"}  # beyond UTF-16's first plane
assert first("def f[\n  T = 1 +](): pass", shapes=True)["type_params"][0][2][1] == ("error", {"text": "1 +"}, [])
# `MISSING` where an expression stands: a `missing` term, the kind tree-sitter expected
missing = first("for in y:\n    pass", shapes=True)
assert missing["target"] == ("missing", {"kind": "identifier"}, []) and kinds(missing["body"]) == ["Pass"]
# Kept in the store and in JSON as any statement and term are
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
    Parse.parse("a = 1\nx = = 2\nfor in y:\n    pass\nz = (1 + )\n", "invalid.py", STORE))))
fresh = new_store()
(again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
with fresh.transaction():
    body = sorted(fresh.singleton(Model.WORKSPACE).files({"path": "invalid.py"}).body, key=lambda e: e.index.get())
    assert [e.child.as_schema().name for e in body] == ["Programs.Python3.Assign", "Programs.Framework.Error",
                                                         "Programs.Python3.For", "Programs.Python3.Assign"]
    assert [(e.line.get(), e.file.as_schema().name) for e in body[1].child.location] == [(2, "Programs.Python3.Module")]
# Only text that is no text raises, before any transaction
fails("x = '\ud800'", "not text: a lone surrogate", 1, 6)
fails("x = 1\n  \udfff", "not text: a lone surrogate", 2, 3)
with STORE.transaction():
    assert "failing.py" not in [e.path.get() for e in STORE.singleton(Model.WORKSPACE).files]  # nothing written
print("ok")